<a href="https://colab.research.google.com/github/msaleem-aisci/deep-learning/blob/main/ANN_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

In [ ]:
X = np.array([[-21,12,14], [8,9,7], [2,9,5], [12,19,15]])

In [ ]:
X.shape

(4, 3)

In [ ]:
class ReLU:
  def forward(self, inputs):
    self.inputs = inputs

    self.output = np.maximum(0, inputs)
    return self.output



class Dense:
    def __init__(self, n_input, n_neurons):
        self.neurons = n_neurons
        self.weights = np.random.randn(n_input, n_neurons) * 0.01
        self.biases = np.zeros((1, n_neurons))


    def forward(self, X):
      self.o = np.dot(X, self.weights) + self.biases

      self.output = ReLU().forward(self.o)


      return self.output


class ANN:
    def __init__(self, layers):
        self.layers = layers
        self.params = {}

        for i in range(len(layers)):  # Start from 1 as the first layer has no incoming weights
            self.params[f'w{i+1}'] = layers[i].weights
            self.params[f'b{i+1}'] = layers[i].biases

    def forward(self, X):
      input_data = X
      for i in range(len(self.layers)):
        input_data = self.layers[i].forward(input_data)

      return input_data

ann = ANN([
    Dense(X.shape[1], 3),  # Input layer to hidden layer
    Dense(3, 3),           # Hidden layer to output layer
])

ann.forward(X)




array([[0.        , 0.00286192, 0.00050221],
       [0.        , 0.00297186, 0.0005215 ],
       [0.        , 0.00204817, 0.00035941],
       [0.        , 0.00598047, 0.00104946]])

In [ ]:
class ANN:
    def __init__(self, layers):
        self.layers = layers
        self.params = {}

        for i in range(len(layers)):  # Start from 1 as the first layer has no incoming weights
            self.params[f'w{i+1}'] = layers[i].weights
            self.params[f'b{i+1}'] = layers[i].biases

    def forward(self, X):
      input_data = X
      for i in range(len(self.layers)):
        input_data = self.layers[i].forward(input_data)

      return input_data

ann = ANN([
    Dense(X.shape[1], 3),  # Input layer to hidden layer
    Dense(3, 3),           # Hidden layer to output layer
])

ann.forward(X)



[[-0.8834776  -5.7436238   1.37751762]
 [ 6.26908824  8.21080562  9.83377696]
 [ 4.4499403   4.04377454  5.34230321]
 [11.79612237 14.58349125 18.55678003]]
[[ 0.          0.          1.37751762]
 [ 6.26908824  8.21080562  9.83377696]
 [ 4.4499403   4.04377454  5.34230321]
 [11.79612237 14.58349125 18.55678003]]
[[ 0.3855325   1.32069258  0.37159149]
 [ 9.91127148 15.14006951  7.72195387]
 [ 6.05865691  8.22288727  4.46002808]
 [18.41273117 28.07621164 14.26300251]]
[[ 0.3855325   1.32069258  0.37159149]
 [ 9.91127148 15.14006951  7.72195387]
 [ 6.05865691  8.22288727  4.46002808]
 [18.41273117 28.07621164 14.26300251]]


array([[ 0.3855325 ,  1.32069258,  0.37159149],
       [ 9.91127148, 15.14006951,  7.72195387],
       [ 6.05865691,  8.22288727,  4.46002808],
       [18.41273117, 28.07621164, 14.26300251]])

In [ ]:
import numpy as np

class ReLU:
    def forward(self, inputs):
        self.inputs = inputs
        self.output = np.maximum(0, inputs)
        return self.output

    def backward(self, d_output):
        d_inputs = d_output.copy()
        d_inputs[self.inputs <= 0] = 0
        return d_inputs

class Dense:
    def __init__(self, n_input, n_neurons, activation=True):
        self.weights = np.random.randn(n_input, n_neurons) * np.sqrt(2.0 / n_input)  # He Initialization
        self.biases = np.zeros((1, n_neurons))
        self.use_activation = activation
        if activation:
            self.activation = ReLU()

    def forward(self, X):
        self.inputs = X
        self.o = np.dot(X, self.weights) + self.biases
        self.output = self.activation.forward(self.o) if self.use_activation else self.o
        return self.output

    def backward(self, d_output, learning_rate):
        if self.use_activation:
            d_output = self.activation.backward(d_output)

        print("Shape: ",d_output.shape)
        print("inout: ",self.inputs.shape)
        dW = np.dot(self.inputs.T, d_output)
        dB = np.sum(d_output, axis=0, keepdims=True)  # Ensure bias remains (1, n_neurons)
        dX = np.dot(d_output, self.weights.T)

        self.weights -= learning_rate * dW
        self.biases -= learning_rate * dB  # Ensure correct shape

        return dX

class ANN:
    def __init__(self, layers, learning_rate=0.1):
        self.layers = layers
        self.learning_rate = learning_rate

    def forward(self, X):
        input_data = X
        for layer in self.layers:
            input_data = layer.forward(input_data)
        return input_data

    def backward(self, d_loss):
        grad = d_loss
        for layer in reversed(self.layers):
            grad = layer.backward(grad, self.learning_rate)
            break

    def train(self, X, y, epochs=1):
        for epoch in range(epochs):
            y_pred = self.forward(X)


            y_pred = y_pred.reshape(y.shape)
            res = y - y_pred
            loss = np.mean((res) ** 2)
            d_loss = -2 * (res) / len(y)


            self.backward(d_loss)

            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Loss: {loss}")

# Example: Works with any output size, including 1 neuron
X = np.random.randn(10, 2)  # 100 samples, 2 features
y = np.random.randn(10, 1)  # 1 output neuron for regression

ann = ANN([
    Dense(X.shape[1], 4),
    Dense(4, 1, activation=False),
])

ann.train(X, y)


Shape:  (10, 1)
inout:  (10, 4)
Epoch 0, Loss: 1.3537069008869387


In [ ]:
t = {1,1}

print(t)

{1}
